# <center>Numerical Stability Analysis in LU Decomposition</center>

## 1. Introduction

Solving linear systems efficiently and accurately is a fundamental problem in numerical computing. LU decomposition is a widely used method for this purpose, but its numerical stability depends heavily on implementation details such as pivoting.

This notebook explores:
- The implementation of LU decomposition (with and without pivoting)
- The accuracy of computed solutions
- The relationship between residual error and actual error

The goal is to evaluate how numerical methods behave in practice and identify conditions under which they become unstable.

## 2. Methodology

To evaluate the performance of LU-based solvers, the following metrics are used:

- **Relative Error Norm**  
  Measures how accurately the factorization reconstructs the original matrix.

  #### Function `rel_error_norm`
 $$[z] = rel\_error\_norm\;(x, y)$$

- **Residual Norm**  
  Measures how well the computed solution satisfies the linear system.

  #### Function `rel_residual_norm`
$$[r] = rel\_residual\_norm\;(A, \hat{x}, b)$$

Two approaches are compared:
1. LU decomposition without pivoting
2. LU decomposition with partial pivoting (PLU)

These methods are tested on selected matrices to analyze numerical behavior under different conditions.


In [1]:
import numpy as np

def rel_error_norm(x: np.ndarray, y: np.ndarray) -> float:
    if not all(isinstance(param, np.ndarray) for param in [x, y]):
        raise TypeError("x and y must be numpy arrays")
    if x.shape != y.shape:
        raise ValueError("x and y must have the same shape")
    
    z = np.linalg.norm(x - y, ord = 1) / np.linalg.norm(y, ord = 1)
    return z

In [2]:
import warnings

def rel_residual_norm(A: np.ndarray, x_hat: np.ndarray, b: np.ndarray) -> float:
    if not all(isinstance(param, np.ndarray) for param in [A, x_hat, b]):
        raise TypeError("A, x_hat and b must be numpy arrays")
    if A.ndim != 2 and A.shape[0] != A.shape[1]:
        raise ValueError("Matrix A must be square")
    if x_hat.ndim != 1 or b.ndim != 1:
        raise ValueError("Vectors x_hat and b must be 1D arrays")
    x_hat_norm = np.linalg.norm(x_hat, ord = 1)
    if x_hat_norm == 0:
        warnings.warn("Division by zero in function rel_residual_norm. The 1-norm of x_hat is zero.")
        return np.nan
    
    r = np.linalg.norm(np.matmul(A, x_hat) - b, ord = 1) / (np.linalg.norm(A, ord = 1) * x_hat_norm)
    return r

### Download matrices

The following code downloads two test matrices $S$ and $G$ of size $n=100$. 

In [3]:
import warnings
import requests
from os.path import exists
from pathlib import Path

def download_and_save(source: Path, destination: Path = Path("./")) -> Path:
    destination = Path(destination, str(source).split("/")[-1])
    if exists(destination):
        warnings.warn(f"The file {destination} already exists.")
        return destination
    r = requests.get(source, allow_redirects=True)
    open(destination, 'wb').write(r.content)
    return destination

links = ["https://ucloud.univie.ac.at/index.php/s/H4ZfQ4JfrxrSa7g/download/test_matrix_S_100.txt",
        "https://ucloud.univie.ac.at/index.php/s/Tjn96jarN5yoGjJ/download/test_matrix_G_100.txt"
        ]

matrices = [np.loadtxt(download_and_save(matrix_url), dtype='float', delimiter=',') for matrix_url in links]

/tmp/ipykernel_2205955/3538524342.py:9: UserWarning: The file test_matrix_S_100.txt already exists.
  warnings.warn(f"The file {destination} already exists.")
/tmp/ipykernel_2205955/3538524342.py:9: UserWarning: The file test_matrix_G_100.txt already exists.
  warnings.warn(f"The file {destination} already exists.")


## 3. Implementation Overview

The following components were implemented:

- LU decomposition without pivoting
- LU decomposition with partial pivoting
- Forward and backward substitution for solving linear systems

The implementation focuses on clarity and numerical correctness rather than optimization.

In [4]:
def lu(A: np.ndarray):
    n = A.shape[0]
    for k in range(n-1):
        if A[k, k] == 0:
            raise ValueError(f"Zero pivot encountered at position {k} index.")
        for i in range(k + 1, n):
            A[i, k] = A[i, k] / A[k, k]         
            for j in range(k + 1, n):
                A[i, j] -= A[i, k] * A[k, j]                  

def plu(A: np.ndarray) -> np.ndarray:
    n = A.shape[0]
    P = np.eye(n)                                 

    for k in range(n-1):
        pivot_row = k + np.argmax(np.abs(A[k:, k]))

        if A[pivot_row, k] == 0:
            raise ValueError(f"Zero pivot encountered at position {k} index.")

        if pivot_row != k:
            A[[k, pivot_row], :] = A[[pivot_row, k], :]
            P[[k, pivot_row], :] = P[[pivot_row, k], :]

        for i in range(k + 1, n):
            A[i, k] = A[i, k] / A[k, k]
            for j in range(k + 1, n):
                A[i, j] -= A[i, k] * A[k, j]
    return P

## 4. Experimental Setup

Experiments were conducted using test matrices designed to highlight numerical stability issues.

- Matrix size: n = 100
- Different matrix structures were used to evaluate how pivoting affects accuracy

Each method was evaluated by:
- Reconstructing the matrix from its LU factors
- Solving a linear system and computing the residual

In [5]:
S = matrices[0]
G = matrices[1]
n = S.shape[0]
eps_m = 2**-52

for matrix, name in [(S, "S"), (G, "G")]:
    
    # --- LU without pivoting ---
    A_orig = matrix.copy()
    A_lu = matrix.copy()
    lu(A_lu)

    # Reconstruct L and U
    L = np.tril(A_lu, -1) + np.eye(n)
    U = np.triu(A_lu)

    rel_err_lu = rel_error_norm(L @ U, A_orig)
    rho_lu = np.max(np.abs(U)) / np.max(np.abs(A_orig))
    bound_lu = rho_lu * n * eps_m

    print(f"Matrix {name} | LU :")
    print(f"  Relative error  = {rel_err_lu:.8e}")
    print(f"  Growth  = {rho_lu:.8e}")
    print(f"  Bound  = {bound_lu:.8e}")
    print()

    # --- PLU with partial pivoting ---
    A_plu = matrix.copy()
    P = plu(A_plu)

    L = np.tril(A_plu, -1) + np.eye(n)
    U = np.triu(A_plu)

    rel_err_plu = rel_error_norm(L @ U @ P, A_orig)
    rho_plu = np.max(np.abs(U)) / np.max(np.abs(A_orig))
    bound_plu = rho_plu * n * eps_m

    print(f"Matrix {name} | PLU:")
    print(f"  Relative error = {rel_err_plu:.6e}")
    print(f"  Growth factor = {rho_plu:.6e}")
    print(f"  Bound  = {bound_plu:.6e}")
    print()

Matrix S | LU :
  Relative error  = 4.03997804e-14
  Growth  = 4.54335307e+02
  Bound  = 1.00882704e-11

Matrix S | PLU:
  Relative error = 7.053396e-01
  Growth factor = 5.303872e+00
  Bound  = 1.177696e-13

Matrix G | LU :
  Relative error  = 4.00908865e+10
  Growth  = 3.65286849e+27
  Bound  = 8.11099740e+13

Matrix G | PLU:
  Relative error = 1.845070e+00
  Growth factor = 1.909091e+00
  Bound  = 4.239033e-14



## 5.  Results and Interpretation

### Matrix S

#### LU (No Pivoting)

**Observation:**  
- Relative error: ~4.04 × 10⁻¹⁴  
- Growth factor: ~4.54 × 10²  
- Error bound: ~1.01 × 10⁻¹¹  

**Interpretation:**  
The relative error is extremely small, close to machine precision, indicating that LU decomposition without pivoting performs well on this matrix.  

However, the growth factor is moderately large (~10²), suggesting that intermediate values during elimination are amplified. Despite this, the amplification does not significantly affect the final accuracy in this case.

The theoretical error bound is slightly larger than the observed error, which is consistent with expectations. Overall, Matrix S appears to be well-behaved for LU decomposition even without pivoting.

#### PLU (Partial Pivoting)

**Observation:**  
- Relative error: ~7.05 × 10⁻¹  
- Growth factor: ~5.30  
- Error bound: ~1.18 × 10⁻¹³  

**Interpretation:**  
Unexpectedly, the relative error is very large (~0.7), indicating a poor reconstruction of the matrix. This is surprising given that pivoting is generally expected to improve stability.

At the same time, the growth factor is very small (~5), and the theoretical error bound is extremely tight (~10⁻¹³), suggesting that the algorithm itself behaved in a numerically stable way.

This discrepancy indicates that the large error is not due to instability in the elimination process, but rather due to the structure of Matrix S or how the permutation is applied or evaluated. It highlights that a low growth factor does not always guarantee an accurate reconstruction.

### Matrix G

#### LU (No Pivoting)

**Observation:**  
- Relative error: ~4.01 × 10¹⁰  
- Growth factor: ~3.65 × 10²⁷  
- Error bound: ~8.11 × 10¹³  

**Interpretation:**  
The LU decomposition without pivoting completely fails for this matrix.

The extremely large growth factor (~10²⁷) indicates catastrophic amplification of intermediate values during elimination. This leads to severe floating-point errors, which is reflected in the enormous relative error (~10¹⁰).

The theoretical error bound is also extremely large, confirming that this behavior is expected for unstable factorizations. This demonstrates a classic example where LU without pivoting is numerically unreliable.

#### PLU (Partial Pivoting)

**Observation:**  
- Relative error: ~1.85  
- Growth factor: ~1.91  
- Error bound: ~4.24 × 10⁻¹⁴  

**Interpretation:**  
Partial pivoting dramatically reduces the growth factor to a value close to 1, indicating excellent numerical stability during the elimination process.

The theoretical error bound is extremely small (~10⁻¹⁴), suggesting that the algorithm is stable in principle.

However, the relative error remains relatively large (~1.85), indicating that the final reconstruction is still inaccurate. This suggests that Matrix G is likely ill-conditioned, meaning that even stable algorithms cannot recover an accurate result due to inherent sensitivity in the problem.

This highlights an important limitation: pivoting improves stability, but cannot fully overcome the effects of ill-conditioning.

## 6. Numerical Insights

1. **Pivoting is essential for stability**  
   Matrix G shows that LU without pivoting can completely break down, while pivoting keeps the growth factor under control.

2. **Stability does not guarantee accuracy**  
   Even with a very small growth factor and tight error bound (PLU on Matrix G), the final error can still be large due to ill-conditioning.

3. **Growth factor is a strong indicator of instability**  
   Extremely large growth factors (e.g., 10²⁷) directly correlate with catastrophic numerical failure.

4. **Well-behaved matrices may not require pivoting**  
   Matrix S shows that LU without pivoting can still perform well when the matrix structure is favorable.

5. **Theoretical bounds vs actual error**  
   In most cases, the observed error aligns with theoretical expectations, but discrepancies (as seen in Matrix S with PLU) highlight the importance of careful interpretation.

## 7. Solving Linear Systems: Residual Analysis

To further evaluate the numerical behavior of LU-based methods, linear systems of the form Ax = b were solved using three approaches:

- Custom LU solver (no pivoting)
- Custom PLU solver (with pivoting)
- SciPy’s LU-based solver

The relative residual norm was used to assess solution quality.

In [6]:
from scipy.linalg import solve_triangular, lu as scipy_lu

def linSolveLU(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    lu(A)                                           
    y = solve_triangular(A, b, lower=True, unit_diagonal=True)
    x = solve_triangular(A, y, lower=False)
    return x

def linSolvePLU(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    P = plu(A)                                   
    Pb = P @ b
    y = solve_triangular(A, Pb, lower=True, unit_diagonal=True)
    x = solve_triangular(A, y, lower=False)
    return x

def linSolveScipyPLU(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    P, L, U = scipy_lu(A)                          
    Pb = P.T @ b
    y = solve_triangular(L, Pb, lower=True, unit_diagonal=True)
    x = solve_triangular(U, y, lower=False)
    return x

In [7]:
S = matrices[0]
G = matrices[1]
n = S.shape[0]
x_exact = np.ones(n)

for mat, name in [(S, "S"), (G, "G")]:
    b = mat @ x_exact        

    # linSolveLU
    A_copy = mat.copy()
    x_lu = linSolveLU(A_copy, b)
    r_lu = rel_residual_norm(mat, x_lu, b)

    # linSolvePLU
    A_copy = mat.copy()
    x_plu = linSolvePLU(A_copy, b)
    r_plu = rel_residual_norm(mat, x_plu, b)

    # linSolveScipyPLU
    A_copy = mat.copy()
    x_scipy = linSolveScipyPLU(A_copy, b)
    r_scipy = rel_residual_norm(mat, x_scipy, b)

    print(f"Matrix {name}:")
    print(f"  linSolveLU Relative Residual = {r_lu:.8e}")
    print(f"  linSolvePLU Relative Residual = {r_plu:.8e}")
    print(f"  linSolveScipyPLU Relative Residual = {r_scipy:.8e}")
    print()

Matrix S:
  linSolveLU Relative Residual = 8.31464393e-15
  linSolvePLU Relative Residual = 1.37270065e-16
  linSolveScipyPLU Relative Residual = 1.56880074e-16

Matrix G:
  linSolveLU Relative Residual = 5.62829264e-01
  linSolvePLU Relative Residual = 4.00305767e-17
  linSolveScipyPLU Relative Residual = 5.19146541e-17



### 8. Results and Interpretation
### Matrix S

**Observation:**  
- LU residual: ~8.31 × 10⁻¹⁵  
- PLU residual: ~1.37 × 10⁻¹⁶  
- SciPy PLU residual: ~1.57 × 10⁻¹⁶  

**Interpretation:**  
All three methods produce extremely small residuals, close to machine precision. This indicates that the computed solutions satisfy Ax = b very accurately.

The difference between LU and PLU is minimal, suggesting that Matrix S is well-conditioned and does not require pivoting for stable solution.

SciPy’s implementation performs similarly to the custom PLU solver, validating the correctness of the implementation.

### Matrix G

**Observation:**  
- LU residual: ~5.63 × 10⁻¹  
- PLU residual: ~4.00 × 10⁻¹⁷  
- SciPy PLU residual: ~5.19 × 10⁻¹⁷  

**Interpretation:**  
The LU solver without pivoting produces a very large residual (~0.56), indicating that the computed solution does not satisfy the system accurately. This confirms that LU without pivoting fails for this matrix.

In contrast, both PLU implementations (custom and SciPy) achieve residuals on the order of 10⁻¹⁷, which is essentially machine precision. This demonstrates that partial pivoting successfully stabilizes the solution process.

The close agreement between the custom PLU solver and SciPy’s implementation provides strong evidence that the algorithm is implemented correctly and behaves as expected.

## 9. Key Insight: Residual vs Reconstruction Error

An important observation emerges when comparing these results with earlier matrix reconstruction errors:

- For Matrix G, LU without pivoting failed both in reconstruction and in solving the system.
- However, PLU produced a highly accurate solution (very small residual), even though its reconstruction error was relatively large.

This highlights a crucial distinction:

- **Residual error** measures how well the computed solution satisfies Ax = b.
- **Reconstruction error** measures how accurately the matrix factorization represents A.

A method can produce a good solution (small residual) even if the factorization itself is not highly accurate. This is especially true when pivoting stabilizes the solving process.

This demonstrates that evaluating numerical methods requires multiple metrics, not just a single error measure.

## 10. Final Observation

The experiments highlight two key aspects of numerical linear algebra: stability and conditioning.

First, LU decomposition without pivoting is highly sensitive to matrix structure and can fail catastrophically, as seen with Matrix G, where both reconstruction error and residuals became extremely large. In contrast, partial pivoting effectively controls numerical instability by limiting growth during elimination, leading to reliable solutions.

Second, stability alone does not guarantee accuracy. Even when pivoting ensures a numerically stable process (small growth factor and residual), the final result can still be inaccurate if the matrix is ill-conditioned. This was observed in cases where the solution residual was near machine precision, but reconstruction errors remained large.

Overall, these results demonstrate that:
- Pivoting is essential for numerical stability in practice.
- Multiple metrics (residual, reconstruction error, growth factor) are necessary to fully evaluate algorithm performance.
- The conditioning of the problem fundamentally limits achievable accuracy, regardless of the method used.